# OCR + Caption Extraction for PoliMemeDecode

Run this notebook on Kaggle to extract Bangla/English text from each meme and generate a visual caption. It produces two CSVs:
- `ocr_caption_train.csv`: Image_name, ocr_text, caption
- `ocr_caption_test.csv`: Image_name, ocr_text, caption

The code stays within open-source tools (EasyOCR + BLIP captioning) and runs fully locally in the Kaggle environment.

## Setup
1. Attach the competition dataset as an input to the notebook (folder containing `Train/` and `Test/`).
2. Adjust `BASE_DIR` below if needed (default assumes the dataset is in `../input/...`).
3. Run all cells. On first run, models will download (~1–2 GB) into the session cache.

In [1]:
# If the packages are already present in your Kaggle image, you can skip this cell.
# Kaggle's PyTorch GPU image usually has torch/torchvision already.
%pip install -q easyocr timm sentencepiece "transformers>=4.42.3" "accelerate>=0.30.1"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json
import pandas as pd
from PIL import Image
import torch
from tqdm import tqdm
import easyocr
from transformers import pipeline

torch.set_grad_enabled(False)

C:\Users\ij4hidu1\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Locate the dataset root. Adjust BASE_DIR manually if auto-detect fails.
# Kaggle datasets usually appear under /kaggle/input.
from pathlib import Path


def find_base_dir() -> Path:
    """Find folder containing Train/Train.csv and Test/Test.csv."""
    search_roots = [
        Path('/kaggle/input/polimemedecode-humor-that-speaks-politics'),
        Path('/kaggle/input/polimemedecode-humor-that-speaks-politics/upload'),
        Path('/kaggle/input'),
        Path('.'),
    ]
    for root in search_roots:
        if not root.exists():
            continue
        if (root / 'Train' / 'Train.csv').exists() and (root / 'Test' / 'Test.csv').exists():
            return root
        for child in root.iterdir():
            if child.is_dir() and (child / 'Train' / 'Train.csv').exists() and (child / 'Test' / 'Test.csv').exists():
                return child
    raise FileNotFoundError(
        'Could not find Train/Train.csv and Test/Test.csv. Set BASE_DIR manually to the dataset folder.'
    )


BASE_DIR = find_base_dir()
print('Using dataset at:', BASE_DIR)

TRAIN_CSV = BASE_DIR / 'Train' / 'Train.csv'
TEST_CSV = BASE_DIR / 'Test' / 'Test.csv'
TRAIN_IMG_DIR = BASE_DIR / 'Train' / 'Image'
TEST_IMG_DIR = BASE_DIR / 'Test' / 'Image'

# Output files (written to the current working directory)
OUT_TRAIN = Path('ocr_caption_train.csv')
OUT_TEST = Path('ocr_caption_test.csv')


In [4]:
# Initialize OCR and caption models
# EasyOCR supports Bangla with language code 'bn'; include 'en' for mixed text.
reader = easyocr.Reader(['bn', 'en'], gpu=torch.cuda.is_available())

# BLIP captioning pipeline (open-source). Using the base model for speed.
device = 0 if torch.cuda.is_available() else -1
captioner = pipeline(
    'image-to-text',
    model='Salesforce/blip-image-captioning-base',
    device=device,
    max_new_tokens=32,
)
print('Using device:', 'cuda' if torch.cuda.is_available() else 'cpu')

Using CPU. Note: This module is much faster with a GPU.
C:\Users\ij4hidu1\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


ValueError: Could not load model Salesforce/blip-image-captioning-base with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForVision2Seq'>, <class 'transformers.models.blip.modeling_blip.BlipForConditionalGeneration'>). See the original errors:

while loading with AutoModelForVision2Seq, an error is thrown:
Traceback (most recent call last):
  File "C:\Users\ij4hidu1\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\pipelines\base.py", line 293, in infer_framework_load_model
    model = model_class.from_pretrained(model, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ij4hidu1\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\models\auto\modeling_auto.py", line 2289, in from_pretrained
    return super().from_pretrained(pretrained_model_nam

In [ ]:
def extract_ocr_text(img_path: Path) -> str:
    """Run EasyOCR and return concatenated text lines."""
    try:
        results = reader.readtext(str(img_path), detail=0, paragraph=True)
        text = ' '.join(results).strip()
    except Exception as exc:
        text = f''
        print(f'[WARN] OCR failed for {img_path.name}: {exc}')
    return text


def generate_caption(img_path: Path) -> str:
    """Generate a brief caption with BLIP."""
    try:
        outputs = captioner(Image.open(img_path).convert('RGB'))
        caption = outputs[0]['generated_text'].strip()
    except Exception as exc:
        caption = ''
        print(f'[WARN] Caption failed for {img_path.name}: {exc}')
    return caption


def process_split(csv_path: Path, img_dir: Path, out_path: Path):
    df = pd.read_csv(csv_path)
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        name = row['Image_name']
        img_path = img_dir / name
        ocr_text = extract_ocr_text(img_path)
        caption = generate_caption(img_path)
        records.append({'Image_name': name, 'ocr_text': ocr_text, 'caption': caption})
    out_df = pd.DataFrame(records)
    out_df.to_csv(out_path, index=False)
    print(f'Wrote {len(out_df)} rows to {out_path}')
    return out_df

In [ ]:
# Run extraction. This will take several minutes (OCR + caption for every image).
train_meta = process_split(TRAIN_CSV, TRAIN_IMG_DIR, OUT_TRAIN)
test_meta = process_split(TEST_CSV, TEST_IMG_DIR, OUT_TEST)

# Quick peek at the outputs
display(train_meta.head())
display(test_meta.head())